# Silver cleaning and quarantine
Full snapshot overwrite. Business rules: docs/01_business_rules.md. Không loại Failed/feedback lỗi khỏi Silver.

In [ ]:
run_id = 'manual'
source_folder = 'Source'
bronze_folder = 'Bronze'
reports_folder = 'Reports'
table_schema = 'dbo'
strict_project_snapshot = True

In [ ]:
from pyspark.sql import functions as F, types as T
spark.conf.set("spark.sql.session.timeZone", "UTC")
def save(df, name):
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(name)
def key(*cols):
    return F.sha2(F.to_json(F.struct(*[F.col(c) for c in cols])), 256)

In [ ]:
import re
from pyspark.sql import functions as F, types as T
assert re.fullmatch(r'[A-Za-z0-9_-]+', run_id), 'Invalid run_id'
assert not table_schema or re.fullmatch(r'[A-Za-z_][A-Za-z0-9_]*',table_schema)
def table_name(name):
    assert re.fullmatch(r'[a-z_]+',name)
    return f'{table_schema}.{name}' if table_schema else name
def table(name): return spark.table(table_name(name))
def save(df, name):
    df.write.format('delta').mode('overwrite').option('overwriteSchema','true').saveAsTable(table_name(name))
def bronze_path(kind): return f'Files/{bronze_folder}/{run_id}/{kind}/'
def source_frame(filename):
    return (spark.read.option('header',True).option('inferSchema',False).option('quote','"')
        .option('escape','"').option('mode','FAILFAST').csv(f'Files/{source_folder}/{filename}'))

In [ ]:
raw = spark.read.parquet(bronze_path('sales'))
raw_count = raw.count()
s = raw.dropDuplicates().withColumn('raw_record', F.to_json(F.struct(*[F.col(c) for c in raw.columns])))
unique_count = s.count()
conflicts = s.groupBy('order_id').count().filter('count > 1').select('order_id').withColumn('conflict', F.lit(True))
s = s.join(conflicts, 'order_id', 'left')
customer_schema = 'name STRING, email STRING, phone STRING'
item_schema = 'ARRAY<STRUCT<product_id:STRING,category:STRING,price:DECIMAL(18,2),quantity:INT>>'
s = (s.withColumn('customer', F.from_json('customer_info', customer_schema))
     .withColumn('parsed_items', F.from_json('items', item_schema))
     .withColumn('order_timestamp', F.coalesce(
         F.try_to_timestamp(F.col('order_date'), F.lit("yyyy-MM-dd'T'HH:mm:ss[.SSSSSSSSS]")),
         F.try_to_timestamp(F.col('order_date'), F.lit('dd/MM/yyyy HH:mm'))))
     .withColumn('age_text', F.when(F.col('customer_age')=='twenty','20').when(F.col('customer_age')=='thirty-five','35').otherwise(F.col('customer_age')))
     .withColumn('age', F.expr('try_cast(age_text as int)'))
     .withColumn('age', F.when(F.col('age').between(18,100), F.col('age')))
     .withColumn('amount_usd', F.expr("try_cast(regexp_replace(total_amount, '^[$]', '') as decimal(18,2))"))
     .withColumn('shipping_usd', F.expr('try_cast(shipping_cost as decimal(18,2))'))
     .withColumn('feedback', F.expr('try_cast(feedback_score as decimal(10,2))'))
     .withColumn('points', F.expr('try_cast(loyalty_points as bigint)'))
     .withColumn('payment', F.when(F.lower(F.regexp_replace('payment_method','[ _]',''))=='creditcard','Credit Card').otherwise(F.trim('payment_method')))
     .withColumn('currency', F.upper(F.trim('currency')))
     .withColumn('location', F.coalesce(F.when(F.length(F.trim('location'))>0,F.trim('location')),F.lit('Unknown'))))
bad_items = F.expr("exists(parsed_items, x -> x.product_id is null OR trim(x.product_id) = '' OR x.category is null OR trim(x.category) = '' OR x.price is null OR x.price <= 0 OR x.quantity is null OR x.quantity <= 0)")
def reason(condition, message):
    return F.when(F.coalesce(condition,F.lit(False)), F.lit(message))
s = s.withColumn('error_reason', F.concat_ws(';',
    reason(F.col('order_id').isNull() | (F.length(F.trim('order_id'))==0),'missing_order_id'),
    reason(F.col('conflict'),'conflicting_order_id'),
    reason(F.col('order_timestamp').isNull() | ~F.year('order_timestamp').isin(2024,2025),'invalid_order_date'),
    reason(F.col('customer').isNull() | F.col('customer.name').isNull(),'invalid_customer_json'),
    reason(F.col('parsed_items').isNull() | (F.size('parsed_items')==0) | bad_items,'invalid_items'),
    reason(F.col('amount_usd').isNull() | (F.col('amount_usd')<=0),'invalid_total_amount'),
    reason(F.col('shipping_usd').isNull() | (F.col('shipping_usd')<0),'invalid_shipping'),
    reason(F.col('currency').isNull() | (F.col('currency')!='USD'),'unsupported_currency'),
    reason(F.col('order_status').isNull() | ~F.col('order_status').isin('Completed','Pending','Failed','Refunded'),'invalid_status'),
    reason(F.col('payment').isNull() | ~F.col('payment').isin('Credit Card','PayPal','COD'),'invalid_payment')))
quarantine = s.filter("error_reason != ''").select('order_id','error_reason','raw_record')
clean = s.filter("error_reason = ''")
orders = clean.select('order_id','order_timestamp', F.to_date('order_timestamp').alias('order_date'),
    F.col('customer.name').alias('customer_name'), F.col('customer.email').alias('customer_email'),
    F.col('customer.phone').alias('customer_phone'), F.col('age').alias('customer_age'), 'location',
    F.coalesce(F.when(F.length(F.trim('device_type'))>0,F.trim('device_type')),F.lit('Unknown')).alias('device_type'),
    F.col('payment').alias('payment_method'),'currency', F.trim('discount_code').alias('discount_code'),
    F.col('shipping_usd').alias('shipping_cost_usd'),F.col('amount_usd').alias('total_amount_usd'),
    'order_status',F.col('points').alias('loyalty_points'),F.col('feedback').alias('feedback_score'),
    F.coalesce((F.col('order_status')!='Failed') & F.col('feedback').between(1,5),F.lit(False)).alias('report_eligible'))
items = clean.select('order_id', F.posexplode('parsed_items').alias('item_index','item')).select('order_id','item_index','item.*')
assert raw_count == raw_count-unique_count + orders.count() + quarantine.count()
assert orders.select('order_id').distinct().count() == orders.count()
save(orders,'silver_sales'); save(items,'silver_sales_items'); save(quarantine,'silver_sales_quarantine')
display(quarantine.groupBy('error_reason').count())
print({'raw':raw_count,'duplicates_removed':raw_count-unique_count,'clean':orders.count(),'quarantine':quarantine.count(),'item_lines':items.count()})

In [ ]:
rate_raw = spark.read.parquet(bronze_path('exchange_rate'))
r = rate_raw.dropDuplicates().withColumn('raw_record',F.to_json(F.struct(*[F.col(c) for c in rate_raw.columns])))
r = r.select('raw_record',F.expr('try_cast(year as int)').alias('year'),F.expr('try_cast(month as int)').alias('month'),
    F.upper(F.trim('from_currency')).alias('from_currency'),F.upper(F.trim('to_currency')).alias('to_currency'),
    F.expr('try_cast(exchange_rate as decimal(18,6))').alias('exchange_rate'))
rk = ['year','month','from_currency','to_currency']
dup = r.groupBy(*rk).count().filter('count > 1').select(*rk).withColumn('conflict',F.lit(True))
r = r.join(dup,rk,'left')
r = r.withColumn('error_reason',F.concat_ws(';',
    reason(F.col('year').isNull() | ~F.col('year').isin(2024,2025),'invalid_year'),
    reason(F.col('month').isNull() | ~F.col('month').between(1,12),'invalid_month'),
    reason(F.col('from_currency').isNull() | F.col('to_currency').isNull() | (F.col('from_currency')!='USD') | (F.col('to_currency')!='VND'),'invalid_currency_pair'),
    reason(F.col('exchange_rate').isNull() | (F.col('exchange_rate')<=0),'invalid_rate'),reason(F.col('conflict'),'conflicting_rate_key')))
rq = r.filter("error_reason != ''").select('raw_record','error_reason')
rc = r.filter("error_reason = ''").select(*rk,'exchange_rate')
assert r.count() == rq.count()+rc.count()
save(rc,'silver_exchange_rate'); save(rq,'silver_exchange_rate_quarantine')
assert rc.count()==24, 'Expected complete 2024-2025 rates; inspect quarantine/source'
display(rc.orderBy('year','month'))